In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import time

In [3]:
spark = SparkSession.builder \
    .appName("Spark_Lab") \
    .master("spark://spark-master:7077") \
    .config("spark.ui.port", "4042") \
    .config("spark.executor.instances","2") \
    .config("spark.executor.core","1") \
    .config("spark.executor.memory","1g") \
    .config("spark.cores.max", "2") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/18 15:55:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
sc = spark.sparkContext
rdd = sc.textFile("/opt/spark/work-dir/groupby_data.csv",4)

In [9]:
header = rdd.first()

In [12]:
rdd_split = rdd.filter(
    lambda x: x != header
).map(
    lambda x: x.split(',') 
)

In [14]:
rdd_split.take(5)

[['1', 'C03571', 'HCM', 'Lamp', 'Furniture', '4', '60'],
 ['2', 'C04853', 'Danang', 'Desk', 'Furniture', '1', '300'],
 ['3', 'C09077', 'HCM', 'Laptop', 'Electronics', '2', '1200'],
 ['4', 'C03111', 'Danang', 'Monitor', 'Electronics', '5', '250'],
 ['5', 'C04132', 'CanTho', 'Tablet', 'Electronics', '4', '400']]

In [10]:
quantity_city = rdd_split.map(
    lambda x: (x[2],1)
).reduceByKey(lambda x,y: x + y)

In [11]:
quantity_city.collect()

[('HCM', 49887), ('CanTho', 49998), ('Danang', 50007), ('Hanoi', 50108)]

In [12]:
product_by_city = rdd_split.map(
    lambda x: (x[2], x[3])
).groupByKey()

In [ ]:
product_by_city.mapValues(list).take(2)

In [9]:
sales_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/opt/spark/work-dir/hr_employees.csv")

In [10]:
sales_df.show(5)

+------+---------------+---------+------+----+----------+-----------------+----------+-----------------+
|emp_id|           name|     dept|salary| age| join_date|performance_score|manager_id|employment_status|
+------+---------------+---------+------+----+----------+-----------------+----------+-----------------+
|     1|Jennifer Acosta|       IT|  2824|29.0|2025-08-24|             NULL|         6|         INACTIVE|
|     2|     Jenna Mayo|Marketing|  4811|55.0|2024-07-26|             91.0|        49|         INACTIVE|
|     3| Kristin Haynes|  Finance|  7924|42.0|2023-10-17|             94.0|        17|           ACTIVE|
|     4| Jessica Murray|Marketing|  8527|26.0|2023-08-03|             NULL|        43|         INACTIVE|
|     5|  Denise Campos|       IT|  5741|45.0|2023-08-16|             62.0|        18|           ACTIVE|
+------+---------------+---------+------+----+----------+-----------------+----------+-----------------+
only showing top 5 rows



In [11]:
sales_df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- join_date: string (nullable = true)
 |-- performance_score: double (nullable = true)
 |-- manager_id: integer (nullable = true)
 |-- employment_status: string (nullable = true)

